# Demandes

Ce notebook présente les demandes et les méthodes de conversion de ces demandes pour obtenir des fichiers d'entrées d'MnMS.

## 1 - Environnement

### 1.1 - Dépendances

In [ ]:
# pip

%pip install --upgrade pip

In [ ]:
# installation des dépendances

%pip install --upgrade sys
%pip install --upgrade os

%pip install --upgrade pathlib

%pip install --upgrade importlib

%pip install --upgrade ast

%pip install --upgrade math

%pip install --upgrade random

%pip install --upgrade datetime

%pip install --upgrade matplotlib    

%pip install --upgrade numpy
%pip install --upgrade pandas
%pip install --upgrade geopandas

%pip install --upgrade utm
%pip install --upgrade geopy
%pip install --upgrade cbsodata
%pip install --upgrade folium
%pip install --upgrade plotly

%pip install --upgrade shapely

%pip install --upgrade logging

In [ ]:
# chargement des dépendances

import sys
import os

from pathlib import Path

import importlib

from lxml import etree

import ast

import math 

import random

from datetime import datetime, timedelta

from matplotlib import dates as mdates
import matplotlib.pyplot as plt
from matplotlib.pyplot import hist

import numpy as np
import pandas as pd
import geopandas as gpd

import utm
import geopy
import cbsodata
import folium
from folium import plugins
from folium.plugins import HeatMap
import plotly.graph_objects as go
import plotly.express as px 

from shapely import wkt
from shapely.geometry import Point
from shapely.geometry import Polygon
from shapely.ops import unary_union

import logging

In [ ]:
# configuration du logging

logging.basicConfig(
    level=logging.INFO,  
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

In [ ]:
# chemin relatif depuis ce notebook jusqu'au dossier parent de modules 

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

### 1.2 - Classes

In [ ]:
# import des classes

import modules.Utils as Utils

importlib.reload(Utils)

## 2 - Contexte

la première étape dans l'analyse de sensibilité des hypothèses de construction de fichier de demande pour le simulateur d'MnMS est de convertir en entrées d'MnMS des fichiers de demande non adaptées à MnMS dont les méthodes de construction divergent. Il est convenu que les méthodes de conversion doivent respecter des hypotèses similaires. Il s'agit aussi de calibrer les fichiers obtenus afin d'obtenir des demandes comparables. 

Cette étape de conversion respecte donc les principes suivants : 
- tirage aléatoire uniforme avec un noeud aléatoire de valeur 42 ou egal à l'identifiant de la ligne;
- découpage postal à partir des données de la municipalités d'Amsterdam;
- zone de demande à partir des données de la municipalités d'Amsterdam;
- mêmes points d'entrées/sorties de la zone de demande;
- même fonction de restriction spatiale.

## 3 - Demande AmsterdamCity_7-9

### 3.1 - Fonctions

In [ ]:
# random time in a given period

def random_time(start="07:00:00", end="09:00:00", seed=42):
    """
    Returns a random time in a given period.

    Parameters
    ----------
    start : str
        The start time.
    end : str
        The end time

    Return
    ------
    str
    """
    rng = random.Random(seed)  # local RNG instance
    
    fmt = "%H:%M:%S"
    t_start = datetime.strptime(start, fmt)
    t_end = datetime.strptime(end, fmt)

    # intervalle en secondes
    delta = int((t_end - t_start).total_seconds())

    # tirage uniforme
    rand_seconds = rng.randint(0, delta)

    # retour formaté
    result = t_start + timedelta(seconds=rand_seconds)
    return result.strftime(fmt)

In [ ]:
# Random point in a polygon

def random_point_in_polygon(poly, seed=42):
    rng = np.random.default_rng(seed)  # local RNG
    minx, miny, maxx, maxy = poly.bounds
    while True:
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        p = Point(float(x), float(y))
        if p.within(poly):
            return p

In [ ]:
# find the nearest highway entry or exit for a given point

def nearest_highway(highway_latlon, point_lat, point_lon):
    """
    Find the nearest highway point to a given lat/lon point.

    Parameters
    ----------
    highway_latlon: list of [lat, lon]
    point_lat, point_lon: coordinates in lat/lon
    
    Returns 
    -------
    nearest highway point (lat, lon)
    """
    dist = [(point_lat - h[0])**2 + (point_lon - h[1])**2 for h in highway_latlon]
    min_idx = dist.index(min(dist))
    return highway_latlon[min_idx]

In [ ]:
# Restrained a demand file to a given area

def demand_in_area(df, global_demand_area):
    """
    Filters and adjusts OD-pairs (origin–destination points) based on whether they lie 
    inside or outside a predefined demand area.

    Parameters
    ----------
    df : pandas.DataFrame
        Input demand data with columns:
            - ORIGIN (string "x y")
            - DESTINATION (string "x y")
            - ID, DEPARTURE, SERVICE
        Coordinates must be in EPSG:32631.

    global_demand_area : shapely.geometry.Polygon or MultiPolygon
        Polygon(s) defining the valid demand/city area in EPSG:32631.

    Returns
    -------
    pandas.DataFrame
        A cleaned OD demand dataframe in which:
            - Trips completely outside the area are removed.
            - Trips with one point outside are adjusted by projecting the outside point
              to the nearest highway access node.
            - ORIGIN and DESTINATION columns are updated with corrected coordinates.
        Columns returned:
            ['ID', 'DEPARTURE', 'ORIGIN', 'DESTINATION', 'SERVICE']
    """

    # Highway reference points
    list_long = [4.875118, 4.939276, 4.972082, 4.962623, 4.942704, 4.913261, 4.842213, 4.844446]
    list_lat  = [52.425440, 52.407426, 52.367443, 52.351031, 52.336912, 52.328299, 52.338848, 52.396390]
    highway = [[lat, lon] for lat, lon in zip(list_lat, list_long)]

    # Convert ORIGIN/DEST to shapely Points
    df["ORIGIN_geom"] = df["ORIGIN"].apply(lambda s: Point(*map(float, s.split())))
    df["DEST_geom"] = df["DESTINATION"].apply(lambda s: Point(*map(float, s.split())))

    demand_areas = gpd.read_file("../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp")
    demand_areas = demand_areas.rename(columns={"[('Deletio": "ZONE_ID", "geometry": "GEOMETRY"})
    demand_areas["ZONE_ID"] = demand_areas["ZONE_ID"].astype(int)
    demand_areas = gpd.GeoDataFrame(demand_areas, geometry="GEOMETRY", crs="epsg:28992")
    demand_areas = demand_areas.to_crs("epsg:32631")

    # 3 — convertir en GeoDataFrame (comme avant)
    origin_gdf, destination_gdf = Utils.prepare_gdf(df, "epsg:32631")
    df = origin_gdf.copy()
    df["DESTINATION"] = destination_gdf["DESTINATION"]

    # 4 — Zones des ORIGIN
    df = gpd.GeoDataFrame(df, geometry="ORIGIN", crs="epsg:32631")
    df["ORIGIN_ZONE"] = df.set_geometry("ORIGIN") \
                          .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    # 5 — Zones des DESTINATION
    df = gpd.GeoDataFrame(df, geometry="DESTINATION", crs="epsg:32631")
    df["DESTINATION_ZONE"] = df.set_geometry("DESTINATION") \
                               .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    new_rows = []

    for idx, row in df.iterrows():
        o = row["ORIGIN_geom"]
        d = row["DEST_geom"]

        # -----------------------------
        # NEW → treat NaN zones as OUTSIDE
        # -----------------------------
        origin_zone_ok = not pd.isna(row["ORIGIN_ZONE"])
        dest_zone_ok   = not pd.isna(row["DESTINATION_ZONE"])

        o_in_geom = o.within(global_demand_area)
        d_in_geom = d.within(global_demand_area)

        # Final inside/outside evaluation
        o_in = o_in_geom and origin_zone_ok
        d_in = d_in_geom and dest_zone_ok

        # Case 1 → both outside → skip entirely
        if not o_in and not d_in:
            continue

        # Case 2 → ORIGIN inside, DEST outside → project DEST
        if o_in and not d_in:
            lat, lon = utm.to_latlon(d.x, d.y, 31, 'N')
            lat2, lon2 = nearest_highway(highway, lat, lon)
            x, y, _, _ = utm.from_latlon(lat2, lon2)
            d = Point(x, y)

        # Case 3 → ORIGIN outside, DEST inside → project ORIGIN
        elif not o_in and d_in:
            lat, lon = utm.to_latlon(o.x, o.y, 31, 'N')
            lat2, lon2 = nearest_highway(highway, lat, lon)
            x, y, _, _ = utm.from_latlon(lat2, lon2)
            o = Point(x, y)

        new_rows.append({
            "ID": row.ID,
            "DEPARTURE": row.DEPARTURE,
            "ORIGIN": f"{o.x} {o.y}",
            "DESTINATION": f"{d.x} {d.y}",
            "SERVICE": "PersonalVehicle"
        })

    return pd.DataFrame(new_rows)

### 3.2 - Construction

La demande AmsterdamCity_7-9 est générée à partir de données fournies par la municipalité d'Amsterdam : 
- le fichier *81_trips_prog_VMA_2019.csv* est une matrice origine-destination / accumulation moyenne : chaque ligne représente une combinaison origine-destination spécifique, chaque colonne une catégorie de moyen de transport associé à une période de la journée avec en valeur l'accumulation moyenne. Les origines et destinations correspondent aux zones du découpage postal des Pays-bas;
- le fichier *_VMA45_SEGs2019_v1.csv* contient divers découpages spatiaux et leur centroides respectifs d'une grande zone incluant les Pays-bas et une partie de l'Europe de l'Ouest;
- le fichier *demand_areas.shp* contient le découpage postal des Pays-bas avec, pour chaque zone, la surface qui lui est associée.
- le fichier *amsterdam_inner.shp* contient le découpage postal de la zone de demande (l'interieur du périphérique d'Amsterdam) avec, pour chaque zone, la surface qui lui est associée.
- 
En pratique la demande Amsterdam_7-9 est construite en 3 étapes : 
- la matrice origine-destination (le fichier *81_trips_prog_VMA_2019.csv*) est d'abords restreinte à la catégorie véhicules légers dans la période 7-9 heures. Les accumulation moyenne sont arrondis à l'unité puis les couples origine-destination aux accumulations nulles sont retirées. Pour chaque couple restant est crée un nombre de lignes égal à son accumulation;
- A chaque ligne est ensuite associée un identifiant unique et un horaire entre 7 et 9 heures (les horaires obtenus sont arrondis à la minute);
- La demande est alors restreinte spatialement à la zone de simulation correpondante à la surface à l'intèrieur du périphérique d'Amsterdam. Tout trajet dont l'origine ou la destination est situé en dehors de ces limites se voit attribuée l'entrée (ou la sortie) la plus proche sur le périphérique.  A chaque origine ou destination est associé un point au hasard dans la zone postale dans laquelle ils sont situés (sauf en ce qui concerne les points aux entrées/sorties de la zone).

D'un point de vue plus technique : 
- Les données géographiques originales utilisent le système de projection epsg:28992;
- Les données géographiques des fichiers de demande MnMS utilisent le système de porjection epsg:32631;
- les trajets dont l'origine et la destination se trouvent à l'extérieur de la zone de demande sont retirés.

L'origine précise des fichiers exploités (date, méthodes) est inconnue.

In [ ]:
# chargement de fichiers

# matrice OD
OD = pd.read_csv("../../data/AmsterdamCity/81_trips_prog_VMA_2019.csv", sep=';')
OD = OD[["O", "D", "PAOS_trips"]]

# tableau ZONE_ID - POINT
netherlands_areas = pd.read_csv("../../data/AmsterdamCity/_VMA45_SEGs2019_v1.csv", sep=';', encoding="ISO-8859-1")
netherlands_areas = netherlands_areas[["ZONE_ID","XCOORD","YCOORD"]]
netherlands_areas["GEOMETRY"] = netherlands_areas.apply(lambda row: Point(row["XCOORD"], row["YCOORD"]), axis=1)
netherlands_areas = netherlands_areas.drop(columns=["XCOORD","YCOORD"])
netherlands_areas = gpd.GeoDataFrame(netherlands_areas, geometry="GEOMETRY",crs="EPSG:28992")
netherlands_areas = netherlands_areas.to_crs("EPSG:32631")
netherlands_areas_dict = netherlands_areas.set_index("ZONE_ID")["GEOMETRY"].to_dict()

# fichier zone amsterdam
amsterdam_area = gpd.read_file('../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp') 
amsterdam_area = gpd.GeoDataFrame(amsterdam_area, geometry='geometry', crs="epsg:28992")
amsterdam_area = amsterdam_area.to_crs("epsg:32631")
area = unary_union(amsterdam_area.geometry)

# tableau ZONE_ID - POLYGON
demand_areas = gpd.read_file("../../data/AmsterdamCity/demand_areas/demand_areas.shp")
#demand_areas = gpd.read_file("../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp")
demand_areas = demand_areas.rename(columns={"[('Deletio":"ZONE_ID", "geometry":"GEOMETRY"})
demand_areas["ZONE_ID"] = demand_areas["ZONE_ID"].astype(int)
demand_areas = gpd.GeoDataFrame(demand_areas, geometry="GEOMETRY", crs="epsg:28992")
demand_areas = demand_areas.to_crs("epsg:32631")
demand_areas_dict = demand_areas.set_index("ZONE_ID")["GEOMETRY"].to_dict()

In [ ]:
# construction du fichier de demande globale

demande_amsterdam = OD.copy()

# arrondi des effectifs moyens par couple OD
demande_amsterdam["PAOS_trips"] = demande_amsterdam["PAOS_trips"].round()

# suppression les lignes où n == 0 ou duplication les lignes selon n, réinitialisation de l’index
demande_amsterdam = demande_amsterdam[demande_amsterdam["PAOS_trips"] > 0].copy()
demande_amsterdam = demande_amsterdam.loc[demande_amsterdam.index.repeat(demande_amsterdam["PAOS_trips"])].copy()
demande_amsterdam = demande_amsterdam.reset_index(drop=True)

# association à un identifiant unique 
demande_amsterdam["ID"] = [i for i in range(len(demande_amsterdam))]

# association à un horaire de départ entre 7 et 9h
demande_amsterdam["DEPARTURE"] = demande_amsterdam.apply(lambda row : random_time(start="07:00:00", end="09:00:00", seed=row["ID"]), axis=1)

In [ ]:
# restriction à la zone d'Amsterdam

# association pour chaque origine et destination des coordonnées du centre du découpage géographique correspondant
demande_amsterdam["ORIGIN"] = demande_amsterdam.apply(lambda row : netherlands_areas_dict[row["O"]], axis=1)
demande_amsterdam["DESTINATION"] = demande_amsterdam.apply(lambda row : netherlands_areas_dict[row["D"]], axis=1)

demande_amsterdam = gpd.GeoDataFrame(demande_amsterdam, geometry="ORIGIN", crs="epsg:32631")

demande_amsterdam["ORIGIN_ZONE"] = (
    demande_amsterdam.set_geometry("ORIGIN")
              .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]
)

demande_amsterdam = gpd.GeoDataFrame(demande_amsterdam, geometry="DESTINATION", crs="epsg:32631")

demande_amsterdam["DESTINATION_ZONE"] = (
    demande_amsterdam.set_geometry("DESTINATION")
              .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]
)

demande_amsterdam["ORIGIN"] = demande_amsterdam.apply(lambda row : random_point_in_polygon(demand_areas_dict[row["ORIGIN_ZONE"]], seed=row["ID"]), axis=1)
demande_amsterdam["DESTINATION"] = demande_amsterdam.apply(lambda row : random_point_in_polygon(demand_areas_dict[row["DESTINATION_ZONE"]], seed=42), axis=1)

# suppresion des variables inutiles
demande_amsterdam = demande_amsterdam.drop(columns=["O", "D", "PAOS_trips","ORIGIN_ZONE","DESTINATION_ZONE"])

# formattage des variables ORIGIN  et DESTINATION
demande_amsterdam["ORIGIN"] = demande_amsterdam.apply(lambda row : f'{row["ORIGIN"].x} {row["ORIGIN"].y}', axis=1)
demande_amsterdam["DESTINATION"] = demande_amsterdam.apply(lambda row : f'{row["DESTINATION"].x} {row["DESTINATION"].y}', axis=1)

# trie croissant par DEPARTURE
demande_amsterdam = demande_amsterdam.sort_values(by="DEPARTURE")

demande_amsterdam = demand_in_area(demande_amsterdam, area)
demande_amsterdam.to_csv("../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv", sep=';', index=False)

### 3.3 - Analyse exploratoire

In [ ]:
# demande

chemin_demande_amsterdam = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv"
demande_amsterdam = pd.read_csv(chemin_demande_amsterdam, sep=';')

#### 3.3.1 - Généralitées

In [ ]:
# types et valeurs nulles

demande_amsterdam.info()

In [ ]:
# quelques statistiques

demande_amsterdam.describe(include="all")

#### 3.2.2 - Distributions

##### 3.2.2.2 - Distribution de l'accumulation en fonction du temps

In [ ]:
# Distribution de l'accumulation en fonction du temps pour un intervalle d'une minute

Utils.display_accumulation_time_distribution(chemin_demande_amsterdam, "Demande originale MATSim : distribution de l'accumulation en fonction du temps pour un intervalle d'une minute", "1min")

In [ ]:
# Distribution de l'accumulation en fonction du temps pour un intervalle de cinq minutes

Utils.display_accumulation_time_distribution(chemin_demande_amsterdam, "Demande originale MATSim : distribution de l'accumulation en fonction du temps pour un intervalle de cinq minutes", "5min")

In [ ]:
# Distribution de l'accumulation en fonction du temps pour un intervalle de dix minutes

Utils.display_accumulation_time_distribution(chemin_demande_amsterdam, "Demande originale MATSim : distribution de l'accumulation en fonction du temps pour un intervalle de dix minutes", "10min")

##### 3.2.2.3 Distribution spatio-temporelle des origines et destinations 

In [ ]:
# calcul de l'accumulation max par coordonnée

origines_demande_amsterdam, destinations_demande_amsterdam = Utils.prepare_gdf(demande_amsterdam, "EPSG:32631")
ori_sum = origines_demande_amsterdam.groupby("ORIGIN")["ID"].count()
dest_sum = destinations_demande_amsterdam.groupby("DESTINATION")["ID"].count()
max_size_value = max(ori_sum.max(), dest_sum.max())

In [ ]:
# Distribution spatio-temporelle des origines

Utils.display_origin_time_distribution(chemin_demande_amsterdam, "Demande originale MATSim : Distribution spatio-temporelle des origines", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)

In [ ]:
# Distribution spatio-temporelle des destinations
 
Utils.display_destination_time_distribution(chemin_demande_amsterdam, "Demande originale MATSim : Distribution spatio-temporelle des destinations", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)

#### 3.2.3 - Dispersion

In [ ]:
# Dispersion des temps de départ

Utils.display_boxplot_departure(chemin_demande_amsterdam, "Demande originale MATSim : dispersion des temps de départ")

## 4 - Demande Matsim_7-9

### 4.1 - fonctions

In [ ]:
# Preprocesses the "time" variable

def process_time_variable(df, column="DEPARTURE"):
    """
    Extract the time (HH:MM:SS) from a string, converts in datetime.
    Round to the minute.


    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame whose column to convert.
    column : str
        Name of the column to comvert

    Returns
    -------
    pandas.DataFrame
        The transformed DataFrame
    """
    def replace_24_hour(t):
        if t.startswith("24:"):
            return "00:" + t.split(":", 1)[1]
        return t
    
    df[column] = df[column].apply(replace_24_hour)
    df[column] = df[column].str.extract(r'(\d{2}:\d{2}:\d{2})')[0]
    df[column] = pd.to_datetime(df[column], format='%H:%M:%S', errors='coerce')
    df[column] = df[column].dt.round("min")
    return df

def restore_time_variable(df, column="DEPARTURE"):
    """
    Convert a datetime column back to a string time format HH:MM:SS.
    Restore '24:' prefix if the original time was midnight after rounding.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the time column.
    column : str
        Name of the column to convert.

    Returns
    -------
    pandas.DataFrame
        The transformed DataFrame with the column as string HH:MM:SS.
    """

    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in DataFrame.")

    # Round to nearest minute
    df[column] = df[column].dt.round("min")

    # Convert to string HH:MM:SS
    df[column] = df[column].dt.strftime("%H:%M:%S")

    def restore_24_format(t):
        # If time is exactly midnight "00:xx:yy" AND original was "24:xx:yy"
        # → We cannot detect original directly anymore, so we only restore pure midnight
        if t.startswith("00:") and t != "00:00:00":
            return "24:" + t.split(":", 1)[1]
        return t

    df[column] = df[column].apply(restore_24_format)

    return df

In [ ]:
# build all demand from the output of a amsterdam matsim simulation for all means of transport

def build_all_demand_df(directory):
    """
    Returns a dataframe contains all demand in ascending order of departure time and all listes of origin and destination coordinates using
    as input the output of a amsterdam matsim simulation.

    Parameters
    ----------
    directory : str
        Relative path to the output of a amsterdam matsim simulation.

    Return
    ------
    demandDf : pandas.DataFrame
        The dataframe containing all demand.
    O_x : list
        List of X origin coordinates.
    O_y : list
        List of Y origin coordinates.
    D_x : list
        List of X destination coordinates.
    D_y : list
        List of Y destination coordinates.
    """
    
    # find xml file name
    fname = directory
    # read xml file
    tree = etree.parse(fname)
    root = tree.getroot()  
    # Initialization 
    demand, id_nb, O_x, O_y, D_x, D_y, dep_time, mobility_service = ([] for i in range(8))

    # Iterate through the root node to get the child node 
    for user in tree.xpath("/population/person") :
        ID = user.get('id')
        plan = user.xpath('plan')
        a = plan[0].xpath("activity")   # [0] is necessary 
        leg = plan[0].xpath("leg")
        x1 = []
        y1 = []
        x2 = []
        y2 = []
        time = []
        service = []
        for b in a :      # Iterate through activity in plan
            x1.append(b.get("x"))
            y1.append(b.get('y'))
            x2.append(b.get("x"))
            y2.append(b.get('y'))
        for c in leg :   # Iterate through leg in plan
            if c.get('mode') == 'car':   # travel mode = car 
                service.append('PersonalVehicle')
            else:
                service.append(c.get('mode'))  # get other travel modes
            time.append(c.get('dep_time')) # append departure time

        x1.pop()
        y1.pop()
        x2.pop(0)
        y2.pop(0)
        id_ = [ID]*len(x1)
        id_nb+=id_ # id
        O_x+=x1   # x-coordinate of the origin
        O_y+=y1   # x-coordinate of the destination
        D_x+=x2   # y-coordinate of the origin
        D_y+=y2   # y-coordinate of the destination
        dep_time+=time  # departure time
        mobility_service+=service  # mobility service
    # Create a list of combinasion of 4 columns
    demand = list(zip(id_nb,dep_time,O_x,O_y,D_x,D_y,mobility_service))   
    #Create dataframe from listes
    demandDf = pd.DataFrame(demand, columns = ['ID', 'DEPARTURE', 'O_x', 'O_y', 'D_x', 'D_y', 'SERVICE'])
    # Concatenate and rename coordinates columns  
    demandDf['ORIGIN'] = demandDf.apply(lambda x : '%s %s' % (x['O_x'], x['O_y']), axis=1)
    demandDf['DESTINATION'] = demandDf.apply(lambda x : '%s %s' % (x['D_x'], x['D_y']), axis=1)
    # Delet the four origins coordinates columns
    demandDf = demandDf.drop(labels=['O_x', 'O_y', 'D_x', 'D_y'], axis=1)
    #Mobility = demandDf['SERVICE'].tolist()
    #indices = [i for i, x in enumerate(Mobility) if x == 'PersonalVehicle']
    #demandDf = demandDf.loc[indices]
    demandDf = demandDf[['ID', 'DEPARTURE', 'ORIGIN', 'DESTINATION', 'SERVICE']]
     # switch to incremental order by dep_time
    demandDf = demandDf.sort_values(by = 'DEPARTURE') 
    demandDf = demandDf.reset_index(drop=True)
    return demandDf, O_x, O_y, D_x, D_y

In [ ]:
# build the car demand from the output of a amsterdam matsim simulation for all means of transport

def build_car_users_only_df(directory):
    """
    Returns a dataframe contains the car demand in ascending order of departure time and all listes of origin and destination coordinates using
    as input the output of a amsterdam matsim simulation.

    Parameters
    ----------
    directory : str
        Relative path to the output of a amsterdam matsim simulation.

    Return
    ------
    df : pandas.DataFrame
        The dataframe containing the car demand.
    """
    
    allUserDf, O_x, O_y, D_x, D_y  = build_all_demand_df(directory)
    mobility = allUserDf['SERVICE'].tolist()
    indices = [i for i, x in enumerate(mobility) if x == 'PersonalVehicle']
    df = allUserDf.loc[indices]
    df = df.reset_index(drop=True)
    return df

In [ ]:
# returns a dataframe of demand during this time period : [start_time ,end_time[ 

def demand_in_period(start_time, end_time, demand_df):
    """
    Returns a dataframe contains the a demand restrained at a given time period.

    Parameters
    ----------
    start_time : int
        Start hour of the time period.
    end_time : int
        End hour of the time period.
    demand_df : pandas.DataFrame
        Dataframe containing the demand to cut.
    

    Return
    ------
    df : pandas.DataFrame
        The dataframe containing the car demand.
    """
    
    departure = demand_df["DEPARTURE"].tolist()
    id = demand_df["ID"].tolist()
    o = demand_df["ORIGIN"].tolist()
    d = demand_df["DESTINATION"].tolist()
    s = demand_df["SERVICE"].tolist()

    
    for i in range(len(departure)-1) :
        down = departure[i]
        up = departure[i+1]
        if ((int(down[:2]) == start_time-1) & (int(up[:2]) == start_time)) :
            a = i + 1
        elif ((int(down[:2]) == end_time - 1) & (int(up[:2]) == end_time)) :

            b = i + 1
    departure = departure[a:b]
    id = id[a:b]
    o = o[a:b]
    d = d[a:b]
    s = s[a:b]
    
    demand = list(zip(id,departure,o,d,s))
    df = pd.DataFrame(demand, columns = ['ID', 'DEPARTURE', 'ORIGIN', 'DESTINATION', 'SERVICE'])
    return df

In [ ]:
# find the nearest highway entry or exit for a given point

def nearest_highway(highway_latlon, point_lat, point_lon):
    """
    Find the nearest highway point to a given lat/lon point.

    Parameters
    ----------
    highway_latlon: list of [lat, lon]
    point_lat, point_lon: coordinates in lat/lon
    
    Returns 
    -------
    nearest highway point (lat, lon)
    """
    dist = [(point_lat - h[0])**2 + (point_lon - h[1])**2 for h in highway_latlon]
    min_idx = dist.index(min(dist))
    return highway_latlon[min_idx]

In [ ]:
# Restrained a demand file to a given area

def demand_in_area(df, global_demand_area):
    """
    Filters and adjusts OD-pairs (origin–destination points) based on whether they lie 
    inside or outside a predefined demand area.

    Parameters
    ----------
    df : pandas.DataFrame
        Input demand data with columns:
            - ORIGIN (string "x y")
            - DESTINATION (string "x y")
            - ID, DEPARTURE, SERVICE
        Coordinates must be in EPSG:32631.

    global_demand_area : shapely.geometry.Polygon or MultiPolygon
        Polygon(s) defining the valid demand/city area in EPSG:32631.

    Returns
    -------
    pandas.DataFrame
        A cleaned OD demand dataframe in which:
            - Trips completely outside the area are removed.
            - Trips with one point outside are adjusted by projecting the outside point
              to the nearest highway access node.
            - ORIGIN and DESTINATION columns are updated with corrected coordinates.
        Columns returned:
            ['ID', 'DEPARTURE', 'ORIGIN', 'DESTINATION', 'SERVICE']
    """

    # Highway reference points
    list_long = [4.875118, 4.939276, 4.972082, 4.962623, 4.942704, 4.913261, 4.842213, 4.844446]
    list_lat  = [52.425440, 52.407426, 52.367443, 52.351031, 52.336912, 52.328299, 52.338848, 52.396390]
    highway = [[lat, lon] for lat, lon in zip(list_lat, list_long)]

    # Convert ORIGIN/DEST to shapely Points
    df["ORIGIN_geom"] = df["ORIGIN"].apply(lambda s: Point(*map(float, s.split())))
    df["DEST_geom"] = df["DESTINATION"].apply(lambda s: Point(*map(float, s.split())))

    demand_areas = gpd.read_file("../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp")
    demand_areas = demand_areas.rename(columns={"[('Deletio": "ZONE_ID", "geometry": "GEOMETRY"})
    demand_areas["ZONE_ID"] = demand_areas["ZONE_ID"].astype(int)
    demand_areas = gpd.GeoDataFrame(demand_areas, geometry="GEOMETRY", crs="epsg:28992")
    demand_areas = demand_areas.to_crs("epsg:32631")

    # 3 — convertir en GeoDataFrame (comme avant)
    origin_gdf, destination_gdf = Utils.prepare_gdf(df, "epsg:32631")
    df = origin_gdf.copy()
    df["DESTINATION"] = destination_gdf["DESTINATION"]

    # 4 — Zones des ORIGIN
    df = gpd.GeoDataFrame(df, geometry="ORIGIN", crs="epsg:32631")
    df["ORIGIN_ZONE"] = df.set_geometry("ORIGIN") \
                          .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    # 5 — Zones des DESTINATION
    df = gpd.GeoDataFrame(df, geometry="DESTINATION", crs="epsg:32631")
    df["DESTINATION_ZONE"] = df.set_geometry("DESTINATION") \
                               .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    new_rows = []

    for idx, row in df.iterrows():
        o = row["ORIGIN_geom"]
        d = row["DEST_geom"]

        # -----------------------------
        # NEW → treat NaN zones as OUTSIDE
        # -----------------------------
        origin_zone_ok = not pd.isna(row["ORIGIN_ZONE"])
        dest_zone_ok   = not pd.isna(row["DESTINATION_ZONE"])

        o_in_geom = o.within(global_demand_area)
        d_in_geom = d.within(global_demand_area)

        # Final inside/outside evaluation
        o_in = o_in_geom and origin_zone_ok
        d_in = d_in_geom and dest_zone_ok

        # Case 1 → both outside → skip entirely
        if not o_in and not d_in:
            continue

        # Case 2 → ORIGIN inside, DEST outside → project DEST
        if o_in and not d_in:
            lat, lon = utm.to_latlon(d.x, d.y, 31, 'N')
            lat2, lon2 = nearest_highway(highway, lat, lon)
            x, y, _, _ = utm.from_latlon(lat2, lon2)
            d = Point(x, y)

        # Case 3 → ORIGIN outside, DEST inside → project ORIGIN
        elif not o_in and d_in:
            lat, lon = utm.to_latlon(o.x, o.y, 31, 'N')
            lat2, lon2 = nearest_highway(highway, lat, lon)
            x, y, _, _ = utm.from_latlon(lat2, lon2)
            o = Point(x, y)

        new_rows.append({
            "ID": row.ID,
            "DEPARTURE": row.DEPARTURE,
            "ORIGIN": f"{o.x} {o.y}",
            "DESTINATION": f"{d.x} {d.y}",
            "SERVICE": row.SERVICE
        })

    return pd.DataFrame(new_rows)

In [ ]:
# calibration method

def calibrate(df, target_number = 0, random_state=42):
    """
    Duplicate or remove lines so the number of lines is equal to target_number

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame to modify.
    target_number : int
        target number of lines.
    random_state : int
        Random state.
    Return
    -------
    pandas.DataFrame
        The modified dataframe.
    """

    # check
    if df.empty:
        raise ValueError("Null or empty dataframe.")
    if target_number <= 0:
        raise ValueError("Invalid target number")



    variation_df = df.copy()
    variation_df = process_time_variable(variation_df, "DEPARTURE")
    
    # if delta = 0 → nothing to do
    delta = target_number - len(df)

    if delta == 0:
        variation_df["ID"] = list(range(len(variation_df)))
        variation_df = restore_time_variable(variation_df, "DEPARTURE")
        return variation_df.copy()


    # sampling
    if delta > 0:
        # duplication
        sampled = variation_df.sample(
            n=delta,
            replace=True,
            random_state=random_state
        )
        variation_df = pd.concat([variation_df, sampled], ignore_index=True)

    else:
        # deletion
        n_remove = -delta

        if n_remove >= len(variation_df):
            raise ValueError(
                f"Cannot remove {n_remove} rows: period only contains {len(variation_df)} rows."
            )

        to_remove = variation_df.sample(
            n=n_remove,
            replace=False,
            random_state=random_state
        ).index

        variation_df = variation_df.drop(index=to_remove).reset_index(drop=True)

    # renumber IDs

    variation_df["ID"] = list(range(len(variation_df)))
    variation_df = restore_time_variable(variation_df, "DEPARTURE")
    return variation_df.copy()

In [ ]:
# Random point in a polygon

def random_point_in_polygon(poly, seed=42):
    rng = np.random.default_rng(seed)  # local RNG
    minx, miny, maxx, maxy = poly.bounds
    while True:
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        p = Point(float(x), float(y))
        if p.within(poly):
            return p

In [ ]:
# Return a demand df with origin and destination points randomly spread in their spatial zone, except those whose point values 
# are part of the excluded set.

def spatial_spread(df, threshold=30, seed=42):
    """
    Return a demand df with origin and destination points randomly spread in their spatial zone,
    except those whose point values are part of the excluded set.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataframe to modify.
    treshold : int
        Accumulation treshold to create a set of excluded points.

    Return
    pandas.DataFrame
    """

    if df.empty:
        raise ValueError("Null or empty dataframe.")

    df = df.copy()
    o, _ = Utils.prepare_gdf(df, "epsg:32631")
    # 1 — ORIGINS à exclure
    excl = (
        o.groupby("ORIGIN")["ID"]
          .count()
          .reset_index()
    )
    excl = excl.loc[excl["ID"] > threshold, "ORIGIN"]
    excl = set(excl)

    # 2 — Charger zones
    #demand_areas = gpd.read_file("../../data/AmsterdamCity/demand_areas/demand_areas.shp")
    demand_areas = gpd.read_file("../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp")
    demand_areas = demand_areas.rename(columns={"[('Deletio": "ZONE_ID", "geometry": "GEOMETRY"})
    demand_areas["ZONE_ID"] = demand_areas["ZONE_ID"].astype(int)
    demand_areas = gpd.GeoDataFrame(demand_areas, geometry="GEOMETRY", crs="epsg:28992")
    demand_areas = demand_areas.to_crs("epsg:32631")

    # 3 — convertir en GeoDataFrame (comme avant)
    origin_gdf, destination_gdf = Utils.prepare_gdf(df, "epsg:32631")
    df = origin_gdf.copy()
    df["DESTINATION"] = destination_gdf["DESTINATION"]

    # 4 — Zones des ORIGIN
    df = gpd.GeoDataFrame(df, geometry="ORIGIN", crs="epsg:32631")
    df["ORIGIN_ZONE"] = df.set_geometry("ORIGIN") \
                          .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    # 5 — Zones des DESTINATION
    df = gpd.GeoDataFrame(df, geometry="DESTINATION", crs="epsg:32631")
    df["DESTINATION_ZONE"] = df.set_geometry("DESTINATION") \
                               .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    # dict zone -> polygon
    demand_areas_dict = demand_areas.set_index("ZONE_ID")["GEOMETRY"].to_dict()

    # 6 — Modifier ORIGIN uniquement si ORIGIN ∉ excl
    def modify_origin(row):
        if row["ORIGIN"] in excl:    # garder tel quel
            return row["ORIGIN"]
        return random_point_in_polygon(demand_areas_dict[row["ORIGIN_ZONE"]], row["ID"])

    # 7 — Modifier DESTINATION uniquement si DESTINATION ∉ excl
    def modify_destination(row):
        if row["DESTINATION"] in excl:
            return row["DESTINATION"]
        return random_point_in_polygon(demand_areas_dict[row["DESTINATION_ZONE"]], row["ID"])

    df["ORIGIN"] = df.apply(modify_origin, axis=1)
    df["DESTINATION"] = df.apply(modify_destination, axis=1)

    # Format final
    df["ORIGIN"] = df["ORIGIN"].apply(lambda p: f"{p.x} {p.y}")
    df["DESTINATION"] = df["DESTINATION"].apply(lambda p: f"{p.x} {p.y}")

    df = df.sort_values("DEPARTURE")
    df = df.drop(columns=["SERVICE"])
    df["SERVICE"] = "PersonalVehicle"

    return df.drop(columns=["ORIGIN_ZONE", "DESTINATION_ZONE"], errors="ignore").copy()

### 4.1 - Construction

Matsim_7-9 est une demande générée à partir d'un fichier de sortie de simulation Matsim sur Amsterdam, Matsim étant un simulateur microscopique de trafic. :
Ce fichier de sortie, le fichier  *agent_plans_with_SAV.xml*, est un fichier listant pour chaque individu un ensemble d'activités et leurs trajets (origine, destination, horaire, moyen de transport) associées.

Premièrement, une fonction construit un fichier de demande global (incluant des origines et destinations dans l'ensemble des Pays-bas) en restreignant aux trajets en voitures. Ce fichier est ensuite transformer pour n'inclure que les trajets entre 7 et 9 heures et ceux dont au moins une extrémitée (origine ou destination) se trouve à l'intérieur de la zone de demande. 
Il s'agit alors de calibrer la demande obtenue pour que l'effectif total soit comparable avec celui de la demande AmsterdamCity_7-9. Pour ce faire les lignes du fichier sont dupliquées par tirage aléatoire uniforme ce qui conduit à l'obtention d'une demande avec de fortes concentration sur chaque oriine ou destination. Les origines et destinations sont alors traités pour leur associer un point au hasard dans la zone postale dans laquelle ils sont situés.

In [ ]:
# chargement de fichiers

# sortie matsim
agent_plan_matsim = "../../data/Matsim_7-9/agent_plans_with_SAV.xml" 

# fichier zone amsterdam
amsterdam_area = gpd.read_file('../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp') 
amsterdam_area = gpd.GeoDataFrame(amsterdam_area, geometry='geometry', crs="epsg:28992")
amsterdam_area = amsterdam_area.to_crs("epsg:32631")
area = unary_union(amsterdam_area.geometry)

# fichier demande AmsterdamCity_7-9
chemin_demande_amsterdam = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__0.00__0_0.00.csv"
demande_amsterdam = pd.read_csv(chemin_demande_amsterdam, sep=';')

In [ ]:
# construction du fichier de demande global

demande_matsim = build_car_users_only_df(agent_plan_matsim) 

In [ ]:
# restriction à la période 7-9h

demande_matsim = demand_in_period(7, 9, demande_matsim)

In [ ]:
# restriction à la zone d'Amsterdam

demande_matsim = demand_in_area(demande_matsim, area)

In [ ]:
# sauvegarde

demande_matsim.to_csv("../../inputs/Matsim_7-9/Matsim_7-9_Original__Original/Matsim_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv", sep=';', index=False)

In [ ]:
# affichage des origines de la demande originale matsim

chemin_demande_matsim_originale = "../../inputs/Matsim_7-9/Matsim_7-9_Original__Original/Matsim_7-9_Original__Original__None_None__None_None__None__0.00__0_0.00.csv"
origines_demande_amsterdam, destinations_demande_amsterdam = Utils.prepare_gdf(demande_matsim, "EPSG:32631")
ori_sum = origines_demande_amsterdam.groupby("ORIGIN")["ID"].count()
dest_sum = destinations_demande_amsterdam.groupby("DESTINATION")["ID"].count()
max_size_value = max(ori_sum.max(), dest_sum.max())
Utils.display_origin_time_distribution(chemin_demande_matsim_originale, "Demande originale MATSim : Distribution spatio-temporelle des origines", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)

In [ ]:
# affichage des destinations de la demande originale matsim

chemin_demande_matsim_originale = "../../inputs/Matsim_7-9/Matsim_7-9_Original__Original/Matsim_7-9_Original__Original__None_None__None_None__None__0.00__0_0.00.csv"
origines_demande_amsterdam, destinations_demande_amsterdam = Utils.prepare_gdf(demande_matsim, "EPSG:32631")
ori_sum = origines_demande_amsterdam.groupby("ORIGIN")["ID"].count()
dest_sum = destinations_demande_amsterdam.groupby("DESTINATION")["ID"].count()
max_size_value = max(ori_sum.max(), dest_sum.max())
Utils.display_destination_time_distribution(chemin_demande_matsim_originale, "Demande originale MATSim : Distribution spatio-temporelle des destinations", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)

In [ ]:
# calibration

effectif_amsterdam = len(demande_amsterdam)
demande_matsim = calibrate(demande_matsim, effectif_amsterdam, 42)
demande_matsim = spatial_spread(demande_matsim, 40)

In [ ]:
# sauvegarde

demande_matsim.to_csv("../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv", sep=';', index=False)

In [ ]:
# affichage des origines de la demande originale matsim

chemin_demande_matsim_calibree = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__0.00__0_0.00.csv"
origines_demande_amsterdam, destinations_demande_amsterdam = Utils.prepare_gdf(demande_matsim, "EPSG:32631")
ori_sum = origines_demande_amsterdam.groupby("ORIGIN")["ID"].count()
dest_sum = destinations_demande_amsterdam.groupby("DESTINATION")["ID"].count()
max_size_value = max(ori_sum.max(), dest_sum.max())
Utils.display_origin_time_distribution(chemin_demande_matsim_calibree, "Demande calibrée MATSim : Distribution spatio-temporelle des origines", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)

In [ ]:
# affichage des destinations de la demande originale matsim

chemin_demande_matsim_calibree = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__0.00__0_0.00.csv"
origines_demande_amsterdam, destinations_demande_amsterdam = Utils.prepare_gdf(demande_matsim, "EPSG:32631")
ori_sum = origines_demande_amsterdam.groupby("ORIGIN")["ID"].count()
dest_sum = destinations_demande_amsterdam.groupby("DESTINATION")["ID"].count()
max_size_value = max(ori_sum.max(), dest_sum.max())
Utils.display_destination_time_distribution(chemin_demande_matsim_calibree, "Demande calibrée MATSim : Distribution spatio-temporelle des destination", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)

### 4.2 - Analyse exploratoire

In [ ]:
# demande

chemin_demande_matsim = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv"
demande_matsim = pd.read_csv(chemin_demande_matsim_calibree, sep=';')

#### 4.2.1 - Généralitées

In [ ]:
# types et valeurs nulles

demande_matsim.info()

In [ ]:
# quelques statistiques

demande_matsim.describe(include="all")

#### 4.2.2 - Distributions

##### 4.2.2.2 - Distribution de l'accumulation en fonction du temps

In [ ]:
# Distribution de l'accumulation en fonction du temps pour un intervalle d'une minute

Utils.display_accumulation_time_distribution(chemin_demande_matsim, "Demande originale MATSim : distribution de l'accumulation en fonction du temps pour un intervalle d'une minute", "1min")

In [ ]:
# Distribution de l'accumulation en fonction du temps pour un intervalle de cinq minutes

Utils.display_accumulation_time_distribution(chemin_demande_matsim, "Demande originale MATSim : distribution de l'accumulation en fonction du temps pour un intervalle de cinq minutes", "5min")

In [ ]:
# Distribution de l'accumulation en fonction du temps pour un intervalle de dix minutes

Utils.display_accumulation_time_distribution(chemin_demande_matsim, "Demande originale MATSim : distribution de l'accumulation en fonction du temps pour un intervalle de dix minutes", "10min")

##### 4.2.2.3 Distribution spatio-temporelle des origines et destinations 

In [ ]:
# calcul de l'accumulation max par coordonnée

origines_demande_matsim, destinations_demande_matsim = Utils.prepare_gdf(demande_matsim, "EPSG:32631")
ori_sum = origines_demande_matsim.groupby("ORIGIN")["ID"].count()
dest_sum = destinations_demande_matsim.groupby("DESTINATION")["ID"].count()
max_size_value = max(ori_sum.max(), dest_sum.max())

In [ ]:
# Distribution spatio-temporelle des origines

Utils.display_origin_time_distribution(chemin_demande_matsim, "Demande originale MATSim : Distribution spatio-temporelle des origines", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)

In [ ]:
# Distribution spatio-temporelle des destination
 
Utils.display_destination_time_distribution(chemin_demande_matsim, "Demande originale MATSim : Distribution spatio-temporelle des destinations", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)

#### 4.2.3 - Dispersion

In [ ]:
# Dispersion des temps de départ

Utils.display_boxplot_departure(chemin_demande_matsim, "Demande originale MATSim : dispersion des temps de départ")